# CodeAlpha Data Science Internship: Car Price Prediction
**Author**: Data Science Intern
**Project Repository**: `CodeAlpha_CarPricePrediction`
**Objective**: Predict a used vehicle's selling price using supervised regression with end-to-end preprocessing, feature engineering, cross-model benchmarking, and residual diagnostics.

## 1. Environment & Library Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# Visual styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

## 2. Data Inspection & Cleaning
We load the verified CarDekho dataset and inspect column types, missing values, and target distribution.

In [ ]:
df = pd.read_csv("../data/car_data.csv")
print(f"Dataset Shape: {df.shape}")
print("\nMissing values:")
print(df.isnull().sum())
df.head()

## 3. Exploratory Data Analysis (EDA)
Let us explore how Selling Price correlates with Present Price and Vehicle Age.

In [ ]:
# Feature engineering: derive Vehicle Age
CURRENT_YEAR = 2024
df["Vehicle_Age"] = CURRENT_YEAR - df["Year"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x="Present_Price", y="Selling_Price", hue="Fuel_Type", ax=axes[0])
axes[0].set_title("Selling Price vs Present (Showroom) Price")
axes[0].set_xlabel("Present Price (Lakhs INR)")
axes[0].set_ylabel("Selling Price (Lakhs INR)")

sns.boxplot(data=df, x="Fuel_Type", y="Selling_Price", ax=axes[1])
axes[1].set_title("Price Distribution by Fuel Type")
plt.tight_layout()
plt.show()

## 4. Train / Test Split (Strict Leakage Prevention)

In [ ]:
from src.preprocessing import create_full_pipeline

X = df.drop(columns=["Selling_Price"])
y = df["Selling_Price"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

## 5. Model Training & Comparison

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = []
trained_pipes = {}

for name, model in models.items():
    pipe = create_full_pipeline(model)
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({"Model": name, "MAE": mae, "RMSE": rmse, "R2": r2})
    trained_pipes[name] = pipe

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

## 6. Residual & Predicted vs. Actual Plots

In [ ]:
best_pipe = trained_pipes["Random Forest"]
y_best_pred = best_pipe.predict(X_test)
residuals = y_test - y_best_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test, y_best_pred, alpha=0.7, color="#2563eb")
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel("Actual Selling Price (Lakhs)")
axes[0].set_ylabel("Predicted Selling Price (Lakhs)")
axes[0].set_title("Actual vs. Predicted Price")

axes[1].scatter(y_best_pred, residuals, alpha=0.7, color="#ea580c")
axes[1].axhline(y=0, color="black", linestyle="--", lw=1.5)
axes[1].set_xlabel("Predicted Selling Price (Lakhs)")
axes[1].set_ylabel("Residuals (Actual - Predicted)")
axes[1].set_title("Residuals vs. Fitted Values")
plt.tight_layout()
plt.show()

## 7. Model Serialization & Verification

In [ ]:
joblib.dump(best_pipe, "../models/car_price_model.joblib")
print("Model saved to ../models/car_price_model.joblib")